In [0]:
"""
Cell 1: SilverTransactionCleaner
OOP concept: Single Responsibility — ye class sirf ek kaam karti hai:
raw/bronze data ko clean + validate karna, streaming pipeline mein use hone ke liye.
"""

from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when


class SilverTransactionCleaner:
    """Bronze se aane wale micro-batch ko clean karke Silver-ready banata hai."""

    def clean(self, df: DataFrame) -> DataFrame:
        return (
            df
            .filter(col("amount") > 0)                     # invalid amounts hatao
            .withColumn(
                "balance_diff_orig",
                col("oldbalanceOrg") - col("newbalanceOrig")
            )
            .withColumn(
                "is_balance_mismatch",
                when(
                    col("balance_diff_orig") != col("amount"), 1
                ).otherwise(0)
            )
            .dropDuplicates(["nameOrig", "nameDest", "amount", "step"])
        )


cleaner = SilverTransactionCleaner()
print(" SilverTransactionCleaner ready")

In [0]:
# Bronze table ko batch mode mein read karke Silver table mein likhna

bronze_df = spark.table("workspace.default.bronze_transactions")

silver_df = cleaner.clean(bronze_df)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_transactions")
)

print(" Silver table updated")
spark.sql("SELECT COUNT(*) AS silver_rows FROM workspace.default.silver_transactions").show()

In [0]:
spark.sql("SELECT COUNT(*) AS silver_rows FROM workspace.default.silver_transactions").show()
spark.sql("SELECT is_balance_mismatch, COUNT(*) FROM workspace.default.silver_transactions GROUP BY is_balance_mismatch").show()